In [0]:
#CREATE VOLUME LOG

CATALOG = "airbnb_obs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.monitoring.volume_log (
    snapshop_ts     TIMESTAMP,
    layer           STRING,
    table_name      STRING,
    row_count       BIGINT
) USING DELTA
""")
print(" volume_log ready")



In [0]:
#SNAPSHOT CURRENT ROW COUNTS

from pyspark.sql import functions as F
from datetime import datetime, timezone

def snapshot_volume(layer, tables):
    rows = []
    for t in tables:
        n = spark.table(f"{CATALOG}.{layer}.{t}").count()
        rows.append((datetime.now(timezone.utc), layer, t, int(n)))
    spark.createDataFrame(rows, ["snapshop_ts","layer","table_name","row_count"]) \
         .write.mode("append").saveAsTable(f"{CATALOG}.monitoring.volume_log")
    print(f"✔ snapshotted volumes for {tables}")

snapshot_volume("bronze", ["hosts", "listings", "bookings"])

In [0]:
# FRESHNESS CHECK

def check_freshness(layer, table_name, ts_column="_ingested_at_utc", max_age_hours=26):
    df = spark.table(f"{CATALOG}.{layer}.{table_name}")
    latest = df.agg(F.max(F.col(ts_column)).alias("m")).collect()[0]["m"]
    if latest is None:
        return "FAIL", None, "no timestamp found"
    age_hours = (datetime.now(timezone.utc) - latest.replace(tzinfo=timezone.utc)).total_seconds() / 3600
    status = "PASS" if age_hours <= max_age_hours else "FAIL"
    return status, round(age_hours, 2), f"data is {age_hours:.1f}h old (SLA {max_age_hours}h)"

for t in ["hosts", "listings", "bookings"]:
    status, age, detail = check_freshness("bronze", t)
    print(f"{t}: {status} — {detail}")

In [0]:
# VOLUME ANOMALY DETECTION

def check_volume_anomaly(table_name, drop_pct_threshold=0.30):
    hist = (spark.table(f"{CATALOG}.monitoring.volume_log")
                 .filter(F.col("table_name") == table_name)
                 .orderBy(F.desc("snapshop_ts"))
                 .limit(2)
                 .collect())
    if len(hist) < 2:
        return "PASS", "not enough history yet"
    current, previous = hist[0]["row_count"], hist[1]["row_count"]
    if previous == 0:
        return "PASS", "no prior baseline"
    change = (current - previous) / previous
    if change < -drop_pct_threshold:
        return "FAIL", f"row count dropped {abs(change)*100:.1f}% ({previous}→{current})"
    return "PASS", f"row count change {change*100:+.1f}% ({previous}→{current})"

for t in ["hosts", "listings", "bookings"]:
    status, detail = check_volume_anomaly(t)
    print(f"{t}: {status} — {detail}")

In [0]:
#CHECK POINT
# to see if it fires, let add one broken snapshot manually and re-run

# simulate a volume collapse in bookings, then re-check
from datetime import datetime, timezone
spark.createDataFrame(
    [(datetime.now(timezone.utc), "bronze", "bookings", 400)],
    ["snapshop_ts","layer","table_name","row_count"]
).write.mode("append").saveAsTable(f"{CATALOG}.monitoring.volume_log")

status, detail = check_volume_anomaly("bookings")
print(f"bookings: {status} — {detail}")